# Air Pollution in Algeria

## Setup

In [71]:
from pathlib import Path

import altair as alt
import pandas as pd
import squarify
import attaviz

attaviz.enable()


In [72]:
def clean_names(df):
    """Standardize column names: lowercase, strip, replace spaces with underscores."""
    df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    return df


def find_project_root(marker="pyproject.toml"):
    """Walk up from this notebook's directory until we find the project root."""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

ANTHROPIC_DIR = DATA_DIR / "AI" / "Anthropic"

RAW_FILE = (
    ANTHROPIC_DIR
    / "release_2026_03_24/data/aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv"
)
ONET_FILE = (
    ANTHROPIC_DIR / "release_2025_09_15/data/intermediate/onet_task_statements.csv"
)
SOC_FILE = ANTHROPIC_DIR / "release_2025_09_15/data/intermediate/soc_structure.csv"

COUNTRY_CODE = "DZ"
COUNTRY_NAME = "Algeria"

In [19]:
df = pd.read_csv(RAW_FILE)
dza = df.copy().query("geo_id == @COUNTRY_CODE")

In [ ]:
FONT_SIZE = 11
CHAR_W = FONT_SIZE * 0.62
LINE_H = FONT_SIZE * 1.35
PAD = 6
SAFETY_PX = 4


def _wrap_label(text: str, value: float, dx: float, dy: float) -> str | None:
    """Word-wrap `text` to fit a (dx, dy) rectangle and append the % value.
    Returns None if the rectangle can't safely hold a label + value.
    """
    avail_w = dx - 2 * PAD - SAFETY_PX
    avail_h = dy - 2 * PAD
    if avail_w <= 0 or avail_h <= 0:
        return None

    max_chars = int(avail_w // CHAR_W)
    max_lines = int(avail_h // LINE_H)
    if max_chars < 3 or max_lines < 1:
        return None

    value_str = f"{value:.1f}%"
    if len(value_str) > max_chars:
        return None

    words = text.split()
    # Refuse to render if the longest single word can't fit on a line —
    # avoids visible mid-word truncation (`Computer…`) that still leaks.
    if any(len(w) > max_chars for w in words):
        return None

    lines: list[str] = []
    current = ""
    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_chars:
            current = candidate
        else:
            lines.append(current)
            current = word
            if len(lines) >= max_lines - 1:
                # No more room for label lines; reserve the last line for value
                break
    if current and len(lines) < max_lines - 1:
        lines.append(current)

    if not lines:
        return None
    lines.append(value_str)
    return "\n".join(lines)


def make_treemap(
    data: pd.DataFrame,
    label_col: str,
    value_col: str,
    title: str,
    subtitle: str | None = None,
    width: int = 720,
    height: int = 480,
    drop_not_classified: bool = True,
) -> alt.Chart:
    """Render a treemap from a (label, value) DataFrame using squarify + Altair.

    Labels are word-wrapped to fit each rectangle; rectangles too small to hold
    a label safely keep their tooltip but show no text.
    """
    d = data.copy()
    if drop_not_classified:
        d = d.loc[
            lambda df: (
                ~df[label_col].str.contains("not_classified", case=False, na=False)
            )
        ]
    d = (
        d.loc[lambda df: df[value_col] > 0]
        .sort_values(value_col, ascending=False)
        .reset_index(drop=True)
    )

    norm_values = squarify.normalize_sizes(d[value_col].tolist(), width, height)
    rects = squarify.squarify(norm_values, 0, 0, width, height)
    coords = pd.DataFrame(rects).assign(
        x2=lambda df: df["x"] + df["dx"], y2=lambda df: df["y"] + df["dy"]
    )
    plot_df = pd.concat([d, coords], axis=1)
    plot_df["label"] = plot_df.apply(
        lambda r: _wrap_label(str(r[label_col]), r[value_col], r["dx"], r["dy"]),
        axis=1,
    )

    base = alt.Chart(plot_df).encode(
        x=alt.X("x:Q", axis=None, scale=alt.Scale(domain=[0, width])),
        x2="x2:Q",
        y=alt.Y("y:Q", axis=None, scale=alt.Scale(domain=[0, height], reverse=True)),
        y2="y2:Q",
    )
    rects_layer = base.mark_rect(stroke="white", strokeWidth=2).encode(
        color=alt.Color(f"{label_col}:N", legend=None),
        tooltip=[
            alt.Tooltip(f"{label_col}:N", title="Group"),
            alt.Tooltip(f"{value_col}:Q", title="% of conversations", format=".2f"),
        ],
    )
    text_layer = (
        base.transform_filter("datum.label != null")
        .mark_text(
            align="left",
            baseline="top",
            dx=PAD,
            dy=PAD,
            fontSize=FONT_SIZE,
            lineBreak="\n",
            color="white",
        )
        .encode(x="x:Q", y="y:Q", text="label:N")
    )
    if subtitle is None:
        subtitle = ""
    return (rects_layer + text_layer).properties(
        width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)
    )


def make_bar(
    data: pd.DataFrame,
    label_col: str,
    value_col: str,
    title: str,
    subtitle: str | None = None,
    width: int = 640,
    drop_not_classified: bool = True,
    top_n: int | None = None,
) -> alt.Chart:
    """Label above each bar (editorial style).

    Always readable regardless of label or bar length, but the chart is taller
    because each row reserves space for label + bar.
    """
    d = data.copy()
    if drop_not_classified:
        d = d.loc[
            lambda df: (
                ~df[label_col].str.contains("not_classified", case=False, na=False)
            )
        ]
    d = (
        d.loc[lambda df: df[value_col] > 0]
        .sort_values(value_col, ascending=False)
        .reset_index(drop=True)
    )
    if top_n is not None:
        d = d.head(top_n)

    row_h = 38
    bar_size = 14
    height = max(200, row_h * len(d))
    x_max = d[value_col].max() * 1.02

    plot_df = d.assign(_zero=0.0)
    y_enc = alt.Y(f"{label_col}:N", sort="-x", title=None, axis=None)

    bars = (
        alt.Chart(plot_df)
        .mark_bar(size=bar_size)
        .encode(
            x=alt.X(
                f"{value_col}:Q",
                title="% of conversations",
                scale=alt.Scale(domain=[0, x_max], nice=False),
            ),
            y=y_enc,
            # color=alt.Color(f"{label_col}:N", legend=None),
            tooltip=[
                alt.Tooltip(f"{label_col}:N", title="Group"),
                alt.Tooltip(f"{value_col}:Q", title="%", format=".2f"),
            ],
        )
    )
    label_above = (
        alt.Chart(plot_df)
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=FONT_SIZE,
            color="#222",
            dy=-(bar_size // 2) - 2,
        )
        .encode(x="_zero:Q", y=y_enc, text=f"{label_col}:N")
    )
    pct_at_end = (
        alt.Chart(plot_df)
        .mark_text(
            align="left",
            baseline="middle",
            dx=PAD,
            color="#444",
            fontSize=FONT_SIZE - 1,
        )
        .encode(
            x=f"{value_col}:Q",
            y=y_enc,
            text=alt.Text(f"{value_col}:Q", format=".1f"),
        )
    )
    if subtitle is None:
        subtitle = ""
    return (bars + label_above + pct_at_end).properties(
        width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)
    )


In [156]:
onet = pd.read_csv(ONET_FILE).assign(
    task_normalized=lambda df: df["Task"].str.lower().str.strip(),
    soc_str=lambda df: df["soc_major_group"].astype(int).astype(str).str.zfill(2),
)
task_to_soc = onet.filter(["task_normalized", "soc_str"]).drop_duplicates()

soc_struct = (
    pd.read_csv(SOC_FILE)
    .dropna(subset=["Major Group"])
    .assign(
        soc_str=lambda df: df["soc_major_group"].astype(int).astype(str).str.zfill(2),
        soc_name=lambda df: df["SOC or O*NET-SOC 2019 Title"].str.replace(
            " Occupations", "", regex=False
        ),
    )
)
soc_name_map = dict(zip(soc_struct["soc_str"], soc_struct["soc_name"]))

tasks = dza.copy().query('facet == "onet_task" and variable == "onet_task_pct"')
real = (
    tasks.copy()
    .loc[lambda df: ~df["cluster_name"].isin(["none", "not_classified"])]
    .assign(task_normalized=lambda df: df["cluster_name"].str.lower().str.strip())
)

soc_df = (
    real.merge(task_to_soc, on="task_normalized", how="left")
    .groupby("soc_str", as_index=False)
    .agg(pct=("value", "sum"))
    .assign(soc_name=lambda df: df["soc_str"].map(soc_name_map))
    .sort_values("pct", ascending=False)
    .reset_index(drop=True)
    .drop(columns=["soc_str"])
    # add new row for "not classified" category to make sure the treemap adds up to 100%
    .pipe(
        lambda df: pd.concat(
            [
                df,
                pd.DataFrame(
                    {"soc_name": ["not_classified"], "pct": [100 - df["pct"].sum()]}
                ),
            ],
            ignore_index=True,
        )
    )
)

In [ ]:
soc_treemap = make_treemap(
    soc_df,
    label_col="soc_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME} (Group by job)",
    subtitle="Categorized using O*NET-SOC codes",
)

attaviz.add_caption(
    soc_treemap,
    "Source: Anthropic Economic Index data",
)

alt.VConcatChart(...)

In [158]:
def request_df(level: int) -> pd.DataFrame:
    return (
        dza.loc[
            lambda df: (
                (df["facet"] == "request")
                & (df["variable"] == "request_pct")
                & (df["level"] == level)
            )
        ]
        .filter(["cluster_name", "value"])
        .copy()
        .rename(columns={"value": "pct"})
        .sort_values("pct", ascending=False)
        .reset_index(drop=True)
    )


req_l2 = request_df(2)
req_l1 = request_df(1)


In [159]:
l2_treemap = make_treemap(
    req_l2,
    label_col="cluster_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME}",
    subtitle="Group by request category (L2)",
)

attaviz.add_caption(
    l2_treemap,
    "Source: Anthropic Economic Index data",
)

alt.VConcatChart(...)

In [160]:
l1_treemap = make_treemap(
    req_l1,
    label_col="cluster_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME}",
    subtitle="Group by request category (L1)",
    width=960,
)

attaviz.add_caption(
    l1_treemap,
    "Source: Anthropic Economic Index data",
)

alt.VConcatChart(...)

In [161]:
l2_bar_chart = make_bar(
    req_l2,
    label_col="cluster_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME}",
    subtitle="Group by request category (L2)",
    top_n=20,
)

attaviz.add_caption(
    l2_bar_chart,
    "Source: Anthropic Economic Index data",
)

alt.VConcatChart(...)

In [162]:
l1_bar_chart = make_bar(
    req_l1,
    label_col="cluster_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME}",
    subtitle="Group by request category (L1)",
    top_n=20,
)

attaviz.add_caption(
    l1_bar_chart,
    "Source: Anthropic Economic Index data",
)

alt.VConcatChart(...)